In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rioxarray as rioxr
import scipy.io
import xarray as xr

from western_us_biomass import dir_info
from western_us_biomass.dir_info import dir_figures
from western_us_biomass.make_figures import maps

In [ ]:
def compute_grid_cell_area(ds):
    """
    Approximate area of each grid cell in km², assuming a regular lat/lon grid.
    Uses the actual lat/lon spacing from the dataset.
    """
    lat = ds.lat.values
    lon = ds.lon.values

    # Grid spacing in degrees
    dlat = np.abs(np.diff(lat).mean())
    dlon = np.abs(np.diff(lon).mean())

    # Convert to radians
    lat_rad = np.deg2rad(lat)
    dlat_rad = np.deg2rad(dlat)
    dlon_rad = np.deg2rad(dlon)

    R = 6371.0  # Earth radius in km

    # Area varies with latitude: A = R² * dlon * dlat * cos(lat)
    area_1d = R**2 * dlon_rad * dlat_rad * np.cos(lat_rad)  # shape: (nlat,)

    # Broadcast to 2D (nlat, nlon)
    area_2d = np.tile(area_1d[:, np.newaxis], (1, len(lon)))

    # Return as DataArray with same coords
    area_da = xr.DataArray(
        area_2d,
        coords={"lat": ds.lat, "lon": ds.lon},
        dims=["lat", "lon"],
        attrs={"units": "km²", "long_name": "grid cell area"},
    )
    return area_da


import glob

fnames = glob.glob("archive/cVeg_*_western_us.nc")

tseries_list = []
for fname in fnames:
    print(fname)
    test = xr.open_dataset(fname)
    area = compute_grid_cell_area(test)
    test_mmt = (test["cVeg"] * (area * 1000 * 1000)).sum(dim=["lat", "lon"]) / 1000 / 1e6

    tseries_comparison = test_mmt.groupby("time.year").mean()[155:175]
    tseries_list.append(tseries_comparison)


# Liu et al. 2025
ds = xr.open_dataset(
    "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/raw_data/Liu_2025/subset_cbiotot_m_36yr.nc"
)
ds_spatial = rioxr.open_rasterio(
    "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/raw_data/Liu_2025/"
    + "CONUS_LCMAP_surta_990m.tif"
)
ds = ds.rio.write_crs(ds_spatial.spatial_ref.crs_wkt)
ds["latitude"] = ds_spatial["y"].values
ds["longitude"] = ds_spatial["x"].values
ds = ds.rename({"latitude": "y", "longitude": "x"})
western_states = maps.SHP_WESTERN
western_states = western_states.to_crs(ds_spatial.spatial_ref.crs_wkt)


ds_west = ds.rio.clip(geometries=western_states.geometry)
start = ds_west["cbiotot"][20, 0, :, :]
end = ds_west["cbiotot"][-1, 0, :, :]

mapdata = (end - start).coarsen(x=5, y=5, boundary="pad").mean() / (2021 - 2005)


slope = xr.open_zarr(dir_info.dir_processed + "data_on_ref_grid/1000m/aspect.zarr")
crs = slope["spatial_ref"].crs_wkt

inputs = xr.open_dataset(dir_info.dir_processed + "data_on_ref_grid/1000m/all_variables.nc")
inputs = inputs.rio.write_crs(crs)
FRF = inputs["forest_remaining_forest"]

cbiotot = ds_west["cbiotot"].isel(level=0)
FRF_reprojected = FRF.rio.reproject_match(cbiotot)

start_year = 1985
n_times = cbiotot.sizes["time"]
years = range(start_year, start_year + n_times)

cbiotot = cbiotot.assign_coords(time=list(years)).rename({"time": "year"})

cbiotot_FRF = cbiotot * FRF_reprojected / 100

cbiotot = cbiotot[20:]

cbiotot_FRF = cbiotot_FRF[15:]

In [ ]:
fnames = glob.glob("archive/cVeg_*_western_us.nc")

tseries_list = []
for fname in fnames:
    print(fname)
    test = xr.open_dataset(fname)
    area = compute_grid_cell_area(test)
    test_mmt = (test["cVeg"] * (area * 1000 * 1000)).sum(dim=["lat", "lon"]) / 1000 / 1e6

    tseries_comparison = test_mmt.groupby("time.year").mean()[155:175]
    tseries_list.append(tseries_comparison)

In [ ]:
import pandas as pd

df_all_westwide = pd.read_csv("figure_data/figure_2/OurStudy_western_stocks.csv")
western_stocks = pd.read_csv("figure_data/figure_2/USFS_western_stocks.csv")
western_stocks = western_stocks.rename(columns={"Unnamed: 0": "year", "0": "live_biomass_MMT"})

In [ ]:
biomass_mean_west = df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).median()
biomass_min_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west = (
    df_all_westwide["live_biomass_MMT"].groupby(df_all_westwide["year"]).quantile(0.975)
)

biomass_mean_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).median()
)
biomass_min_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.025)
)
biomass_max_west_delta = (
    df_all_westwide["live_biomass_MMT_delta"].groupby(df_all_westwide["year"]).quantile(0.975)
)

# Li et al. 2025

In [ ]:
# Load the .mat file
trend1 = scipy.io.loadmat(
    "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/raw_data/Li_2025/datasets/trendBC1016.mat"
)
trend2 = scipy.io.loadmat(
    "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/raw_data/Li_2025/datasets/trendBC1622.mat"
)
lat = scipy.io.loadmat(
    "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/raw_data/Li_2025/datasets/lat.mat"
)
lon = scipy.io.loadmat(
    "/dfs10/jranders_lab1/users/czarakas/uncertain_land_sink_data/raw_data/Li_2025/datasets/lon.mat"
)

western_states = maps.SHP_WESTERN

da_trend1 = xr.DataArray(
    trend1["trendBC1016"],
    dims=["lat", "lon"],  # specify dimension names
    coords={
        "lat": lat["lat"].squeeze(),
        "lon": lon["lon"].squeeze(),
    },
).rio.write_crs("EPSG:4326")
da_trend1 = da_trend1.rio.set_spatial_dims(x_dim="lon", y_dim="lat")

da_trend2 = xr.DataArray(
    trend2["trendBC1622"],
    dims=["lat", "lon"],  # specify dimension names
    coords={
        "lat": lat["lat"].squeeze(),
        "lon": lon["lon"].squeeze(),
    },
).rio.write_crs("EPSG:4326")
da_trend1 = da_trend1.rio.set_spatial_dims(x_dim="lon", y_dim="lat")


from shapely.geometry import Point

# Step 1: Bounding box clip (what you already have)
minx, miny, maxx, maxy = western_states.total_bounds
lon_mask = (da_trend1["lon"] >= minx) & (da_trend1["lon"] <= maxx)
lat_mask = (da_trend1["lat"] >= miny) & (da_trend1["lat"] <= maxy)
da_trend1_clipped = da_trend1.where(lon_mask & lat_mask, drop=True)
da_trend2_clipped = da_trend2.where(lon_mask & lat_mask, drop=True)

# Step 2: Create a mask for points inside western_states polygons
# Combine all geometries into one
from shapely.ops import unary_union

western_states_union = unary_union(western_states.geometry)

# Create 2D coordinate arrays
lons_2d, lats_2d = np.meshgrid(da_trend1_clipped["lon"].values, da_trend1_clipped["lat"].values)

# Check which points are inside the shapefile
mask = np.zeros(lons_2d.shape, dtype=bool)
for i in range(lons_2d.shape[0]):
    for j in range(lons_2d.shape[1]):
        point = Point(lons_2d[i, j], lats_2d[i, j])
        mask[i, j] = western_states_union.contains(point)

# Apply the mask
da_trend1_clipped = da_trend1_clipped.where(mask)
da_trend2_clipped = da_trend2_clipped.where(mask)

print(
    "Final clipped lat range:",
    da_trend1_clipped["lat"].min().values,
    "to",
    da_trend1_clipped["lat"].max().values,
)
print(
    "Final clipped lon range:",
    da_trend1_clipped["lon"].min().values,
    "to",
    da_trend1_clipped["lon"].max().values,
)

In [ ]:
# Reproject FRF to WGS84 lat/lon (EPSG:4326)
# FRF_latlon = FRF.rio.reproject("EPSG:4326", resampling=Resampling.bilinear)

# Then reproject_match to snap to da_trend1's exact grid
# FRF_matched = FRF_latlon.rio.reproject_match(da_trend1_clipped, resampling=Resampling.bilinear)

In [ ]:
# FRF_matched_clipped=FRF_matched.where(lon_mask & lat_mask, drop=True)

In [ ]:
gridcell_m2 = 25000 * 25000
m2_per_ha = 10000
gridcell_ha = gridcell_m2 / m2_per_ha

In [ ]:
trend1_westwide = (da_trend1_clipped * gridcell_ha / 1e6).sum(dim=["lat", "lon"])
trend2_westwide = (da_trend2_clipped * gridcell_ha / 1e6).sum(dim=["lat", "lon"])

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(15, 7))
# .rio.clip(western_states.geometry).plot()
plt.subplot(1, 2, 1)
maps.plot_map(
    da_trend1_clipped,
    shp=western_states,
    latlon=False,
    cbar_label="MgC/ha/yr",
    cmap=plt.cm.BrBG,
    savefig=None,
    clims=[-1.5, 1.5],
    title="Linear trend 2010-2016 \n (mean: 0.08 MgC/ha/yr)",
    ax=ax[0],
)
# plt.xlim([-125,-102])
# plt.ylim([31,49.5])

plt.subplot(1, 2, 2)
maps.plot_map(
    da_trend2_clipped,
    shp=western_states,
    latlon=False,
    cbar_label="MgC/ha/yr",
    cmap=plt.cm.BrBG,
    savefig=None,
    clims=[-1.5, 1.5],
    title="Linear trend 2016-2022 \n (mean: -0.28 MgC/ha/yr)",
    ax=ax[1],
)
plt.suptitle("Li et al. 2025 live biomass carbon trends")
plt.tight_layout()
plt.savefig(dir_figures + "Li_2025_western.pdf")
# plt.xlim([-125,-102])
# plt.ylim([31,49.5])

In [ ]:
print(np.nanmean(da_trend1_clipped))
print(np.nanmean(da_trend2_clipped))

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(15, 7))
plt.subplot(1, 2, 1)
maps.plot_map(
    mapdata * 10000 / 1000,
    shp=western_states,
    latlon=False,
    cbar_label="MgC/ha/yr",
    cmap=plt.cm.BrBG,
    savefig=None,
    clims=[-1.5, 1.5],
    ax=ax[0],
    # title="Linear trend 2010-2016 \n (mean: 0.08 MgC/ha/yr)",
)
plt.subplot(1, 2, 2)
tseries = (cbiotot_FRF * 990 * 990 / 1000 / 1e6).sum(dim=["x", "y"])
plt.plot(np.arange(2005, 2021), tseries)
plt.plot(
    np.arange(2005, 2021), tseries.rolling(year=5, center=True).mean(), linestyle="--"
)  # .plot()
# plt.plot(np.arange(2005,2021), (ds_west['cbiotot']*990*990/1000/1e6)[20:,0,:,:].sum(dim=['x','y']))
plt.axvspan(xmin=2005, xmax=2022, alpha=0.2, color="gray")
plt.ylabel("Western live biomass (MMT C)")
plt.tight_layout()
plt.savefig(dir_figures + "Liu_2025_western.pdf")

# Save data

In [ ]:
# Stack all CMIP6 anomaly time series and compute percentile envelope
cmip_anomalies = []
for i, fname in enumerate(fnames):
    tseries_comparison = tseries_list[i]
    anomaly = (tseries_comparison - tseries_comparison[0]) * 0.8
    cmip_anomalies.append(anomaly.values)

cmip_array = np.array(cmip_anomalies)  # shape: (n_models, n_times)
cmip_times = tseries_list[0].year.values

In [ ]:
import xarray as xr

da = xr.DataArray(
    cmip_array,
    dims=["ensemble_member", "year"],
    coords={"year": cmip_times, "ensemble_member": range(22)},
)
df = da.to_dataset(name="change_since_2005")
df.to_netcdf("figure_data/figure_5/CMIP_tseries.nc")

In [ ]:
tseries = (cbiotot_FRF * 990 * 990 / 1000 / 1e6).sum(dim=["x", "y"])
tseries_df = tseries.to_dataset(name="Liu_biomass")
tseries_df.to_netcdf("figure_data/figure_5/Liu_tseries.nc")

In [ ]:
import pandas as pd

trends_df = pd.DataFrame(
    [
        {"year_start": 2010, "year_end": 2016, "trend": trend1_westwide.values},
        {"year_start": 2016, "year_end": 2022, "trend": trend2_westwide.values},
    ]
)
trends_df.to_csv("figure_data/figure_5/Li_trends.csv")